In [ ]:
!pip install datasets transformers accelerate tokenizers

In [ ]:
from transformers import GPT2Tokenizer, GPT2LMHeadModel,DataCollatorForLanguageModeling,Trainer,TrainingArguments

from datasets import load_dataset

In [ ]:
dataset=load_dataset("text", data_files="/content/doctor_patient_dataset.txt", sample_by="paragraph")

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 142
    })
})

In [ ]:
tokenizer=GPT2Tokenizer.from_pretrained("gpt2")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

In [ ]:
tokenizer.pad_token=tokenizer.eos_token

In [ ]:
def tokenize_function(examples):
  tokenizer_inputs=tokenizer(examples["text"],truncation=True,padding="max_length",max_length=128)
  tokenizer_inputs["labels"]=tokenizer_inputs["input_ids"].copy()
  return tokenizer_inputs

In [ ]:
tokenized_dataset=dataset.map(tokenize_function,batched=True,num_proc=4,remove_columns=["text"])

Map (num_proc=4):   0%|          | 0/142 [00:00<?, ? examples/s]

In [ ]:
print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 142
    })
})


In [ ]:
model=GPT2LMHeadModel.from_pretrained("gpt2")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  548MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer,mlm=False)

In [ ]:
training_args=TrainingArguments(
    output_dir="./dr_patient_finetuned",
    num_train_epochs=5,
    per_device_train_batch_size=4,
    save_steps=500,
    save_total_limit=2
)

In [ ]:
trainer=Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=tokenized_dataset["train"]
)

In [ ]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=180, training_loss=2.0763147989908854, metrics={'train_runtime': 1792.7929, 'train_samples_per_second': 0.396, 'train_steps_per_second': 0.1, 'total_flos': 46379335680000.0, 'train_loss': 2.0763147989908854, 'epoch': 5.0})

In [ ]:
model.save_pretrained("./gpt2-dr_pat")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
tokenizer.save_pretrained("./gpt2-dr_pat")

('./gpt2-dr_pat/tokenizer_config.json', './gpt2-dr_pat/tokenizer.json')

In [ ]:
from transformers import pipeline

text_generator = pipeline("text-generation", model="./gpt2-dr_pat")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [ ]:
prompt_doctor="Hello, what are you suffering from?"

In [ ]:
output_doctor=text_generator(prompt_doctor, max_length=50, do_sample=True)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
print(output_doctor[0]['generated_text'])

Hello, what are you suffering from?
Doctor: I'm going to check your blood sugar and cholesterol, and you'll need to take a blood test right away.
Patient: Is that okay?
Doctor: Yes, it's normal. I don't want to take any new blood, but I need to check for something called a high-glutamylase.
Patient: That's a warning sign. It's a common side effect of a certain type of sugar called insulin-like growth factor-I (IGF-I).
Doctor: That's serious stuff. Let me check it right away. Let me check your blood sugar, and check for a clot.
Patient: Is that normal, or is it something that we're really worried about?
Doctor: I'm concerned about a clot that's forming inside your liver, and my test today is showing it's not a clot. This type of clotting is called a liver ache and is a sign of an inflammatory response, which is a condition where the liver starts to clot.
Patient: Does anyone else notice this?
Doctor: Actually, I've noticed a very small amount of my liver ache and some swelling around th

In [ ]:
prompt_patient="I have a fever doctor, what should I do?"

In [ ]:
output_patient=text_generator(prompt_patient, max_length=50, do_sample=True)

[transformers] Both `max_new_tokens` (=256) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


In [ ]:
print(output_patient[0]['generated_text'])

I have a fever doctor, what should I do?
Doctor: I'm having trouble getting enough air to breathe normally, and I'm starting to feel nauseous.
Doctor: Is that common, or is it something you want to address right away?
Patient: My fever, my asthma, my allergies, all have changed a lot, and I don't feel tired or overwhelmed right now.
Doctor: This is really concerning for you. I want to make sure you're well able to respond quickly, and I want you to be able to comfortably breathe normally.
Patient: I understand. I've had a really bad cough for a few days now, and I've been feeling really bad about it for the past two weeks.
Doctor: This pattern, combined with your history of asthma exacerbations, raises concern for a new type of biliary tract infection called biliary cirrhosis.
Patient: Is that really happening?
Doctor: It's happening with a viral infection called Mycobacterium tuberculosis, which can cause bronchoconstrictor, which is inflammation of the bronchioles, and bile ducts.
Pa